In [6]:
# Cell 1 — Imports
# These are all the libraries (tools) we need for the entire project

import numpy as np                          # Maths engine — used for log(), matrix operations
import pandas as pd                         # DataFrames — the main data structure we use
import statsmodels.api as sm                # OLS regression
import scipy.stats as stats                 # Chi-squared distribution for Hausman test
import simfin as sf                         # SimFin bulk data
import yfinance as yf                       # Just for SPY benchmark (market index)
from scipy.stats import linregress          # Simple linear regression — used to compute Beta

from statsmodels.stats.outliers_influence import variance_inflation_factor  # VIF diagnostic
from linearmodels.panel import PanelOLS, RandomEffects                       # Panel estimators

pd.set_option('display.max_rows', None)     # Show all rows when printing DataFrames

print("All imports successful")

All imports successful


In [7]:
# Cell 2 — Configure SimFin
# Tell SimFin where to store downloaded data and which API key to use

SF_API_KEY = 'key'    # <-- paste your actual API key between the quotes

sf.set_api_key(SF_API_KEY)

# This is the folder where SimFin saves the CSV files after downloading
# It creates the folder automatically if it doesn't exist
sf.set_data_dir('~/simfin_data')

print("SimFin configured successfully")

SimFin configured successfully


In [8]:
# Cell 3 — Download SimFin Bulk Data
# SimFin downloads large CSV files and saves them to ~/simfin_data
# The FIRST time you run this it will take a minute or two to download
# Every time after that it loads instantly from the saved files on your computer

print("Downloading income statements...")
income_df = sf.load_income(variant='annual', market='us')

print("Downloading balance sheets...")
balance_df = sf.load_balance(variant='annual', market='us')

print("Downloading cash flow statements...")
cashflow_df = sf.load_cashflow(variant='annual', market='us')

print("Downloading share prices...")
prices_df = sf.load_shareprices(variant='daily', market='us')

print("\nAll datasets loaded successfully")
print(f"Income rows    : {len(income_df):,}")
print(f"Balance rows   : {len(balance_df):,}")
print(f"Cash flow rows : {len(cashflow_df):,}")
print(f"Price rows     : {len(prices_df):,}")

Dataset "us-income-annual" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-annual" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-cashflow-annual" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-shareprices-daily" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!

All datasets loaded successfully
Income rows    : 16,907
Balance rows   : 16,908
Cash flow rows : 16,905
Price rows     : 6,250,153


In [9]:
# Cell 4 — Inspect Column Names
# Before we can build ratios we need to know the exact column names SimFin uses
# This cell just prints them out so we can check spelling before using them

print("=== INCOME STATEMENT COLUMNS ===")
print(income_df.columns.tolist())

print("\n=== BALANCE SHEET COLUMNS ===")
print(balance_df.columns.tolist())

print("\n=== CASH FLOW COLUMNS ===")
print(cashflow_df.columns.tolist())

print("\n=== SHARE PRICE COLUMNS ===")
print(prices_df.columns.tolist())

=== INCOME STATEMENT COLUMNS ===
['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Expenses', 'Selling, General & Administrative', 'Research & Development', 'Depreciation & Amortization', 'Operating Income (Loss)', 'Non-Operating Income (Loss)', 'Interest Expense, Net', 'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)', 'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net', 'Income (Loss) from Continuing Operations', 'Net Extraordinary Gains (Losses)', 'Net Income', 'Net Income (Common)']

=== BALANCE SHEET COLUMNS ===
['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Oth

In [12]:
# Cell 5 — Filter to our 25 tickers and merge (fixed)

TICKERS = [
    'KO', 'XOM', 'JPM', 'PG', 'JNJ',
    'SHEL', 'HSBC', 'AZN', 'UL', 'BTI',
    'MCD', 'WMT', 'CVX', 'PFE', 'MRK',
    'T', 'VZ', 'IBM', 'GE', 'CAT',
    'MMM', 'D', 'SO', 'BAC', 'WFC'
]

# Reset index first — this moves Ticker and Report Date from the index
# into regular columns so we can filter and merge on them normally
inc = income_df.reset_index()
bal = balance_df.reset_index()
cf  = cashflow_df.reset_index()

# Filter to our 25 tickers
inc = inc[inc['Ticker'].isin(TICKERS)].copy()
bal = bal[bal['Ticker'].isin(TICKERS)].copy()
cf  = cf[cf['Ticker'].isin(TICKERS)].copy()

# Income and cashflow use 'FY', balance sheet uses 'Q4' for the same annual row
# So we filter each one differently
inc = inc[inc['Fiscal Period'] == 'FY']
cf  = cf[cf['Fiscal Period'] == 'FY']
bal = bal[bal['Fiscal Period'] == 'Q4']   # <-- balance sheet annual rows are labelled Q4

# Keep only years 2009 to 2024
for df in [inc, bal, cf]:
    df.drop(df[(df['Fiscal Year'] < 2009) | (df['Fiscal Year'] > 2024)].index, inplace=True)

# Merge all three on Ticker + Fiscal Year
fundamentals = pd.merge(inc, bal, on=['Ticker', 'Fiscal Year'], how='inner', suffixes=('_inc', '_bal'))
fundamentals = pd.merge(fundamentals, cf, on=['Ticker', 'Fiscal Year'], how='inner', suffixes=('', '_cf'))

print(f"Merged fundamentals shape: {fundamentals.shape}")
print(f"Tickers found: {sorted(fundamentals['Ticker'].unique().tolist())}")
print(f"Years covered: {sorted(fundamentals['Fiscal Year'].unique().tolist())}")

Merged fundamentals shape: (72, 82)
Tickers found: ['CAT', 'CVX', 'D', 'GE', 'IBM', 'JNJ', 'KO', 'MCD', 'MMM', 'MRK', 'PFE', 'PG', 'T', 'WMT', 'XOM']
Years covered: [2020, 2021, 2022, 2023, 2024]


In [20]:
# Cell 6 — Build df_simfin from the 15 tickers SimFin has

df_simfin = pd.DataFrame()

df_simfin['Company']        = fundamentals['Ticker'].values
df_simfin['Year']           = fundamentals['Fiscal Year'].values
df_simfin['Net_Income']     = fundamentals['Net Income'].values
df_simfin['Total_Assets']   = fundamentals['Total Assets'].values
df_simfin['Total_Equity']   = fundamentals['Total Equity'].values
df_simfin['Dividends_Paid'] = fundamentals['Dividends Paid'].abs().values
df_simfin['Shares']         = fundamentals['Shares (Diluted)_inc'].values

# Replace zeros with NaN to avoid division by zero
df_simfin['Net_Income']     = df_simfin['Net_Income'].replace(0, np.nan)
df_simfin['Total_Assets']   = df_simfin['Total_Assets'].replace(0, np.nan)
df_simfin['Total_Equity']   = df_simfin['Total_Equity'].replace(0, np.nan)

# Engineer the ratios
df_simfin['DPR']        = df_simfin['Dividends_Paid'] / df_simfin['Net_Income']
df_simfin['DivYield']   = df_simfin['Dividends_Paid'] / df_simfin['Total_Assets']
df_simfin['ROE']        = df_simfin['Net_Income']     / df_simfin['Total_Equity']
df_simfin['Log_Assets'] = np.log(df_simfin['Total_Assets'])

# Beta placeholder — we fill this properly in a later cell
df_simfin['Beta']       = np.nan

df_simfin.reset_index(drop=True, inplace=True)

print(f"df_simfin shape: {df_simfin.shape}")
print(f"Companies: {sorted(df_simfin['Company'].unique().tolist())}")
print(f"Years: {sorted(df_simfin['Year'].unique().tolist())}")
print(df_simfin[['Company','Year','DPR','DivYield','ROE','Log_Assets']].head(10).to_string())

df_simfin shape: (72, 12)
Companies: ['CAT', 'CVX', 'D', 'GE', 'IBM', 'JNJ', 'KO', 'MCD', 'MMM', 'MRK', 'PFE', 'PG', 'T', 'WMT', 'XOM']
Years: [2020, 2021, 2022, 2023, 2024]
  Company  Year       DPR  DivYield       ROE  Log_Assets
0     CAT  2020  0.748165  0.028637  0.194954   25.084120
1     CAT  2021  0.359377  0.028167  0.392892   25.139609
2     CAT  2022  0.363908  0.029777  0.421937   25.129290
3     CAT  2023  0.247992  0.029299  0.529918   25.194630
4     CAT  2024  0.245182  0.030149  0.553606   25.197917
5     CVX  2020 -1.741115  0.040248 -0.041763   26.203029
6     CVX  2021  0.651456  0.042495  0.111655   26.201965
7     CVX  2022  0.309263  0.042560  0.221322   26.275097
8     CVX  2023  0.530488  0.043328  0.131965   26.290205
9     CVX  2024  0.668195  0.045929  0.115313   26.272101


In [21]:
# Cell 7 — Scrape the 10 missing tickers from yfinance
# These are the firms SimFin doesn't cover on the free tier
# We use the same scraping logic as your original notebook

MISSING_TICKERS = ['AZN', 'BAC', 'BTI', 'HSBC', 'JPM', 'SHEL', 'SO', 'UL', 'VZ', 'WFC']

# Years to collect — matching the SimFin range
TARGET_YEARS = [2020, 2021, 2022, 2023, 2024]

all_yf_data = []

for t in MISSING_TICKERS:
    print(f"\nProcessing {t}...")

    try:
        stock = yf.Ticker(t)

        # Pull annual financial statements
        financials    = stock.get_income_stmt(freq='yearly', pretty=False).T
        balance_sheet = stock.get_balance_sheet(freq='yearly', pretty=False).T
        cash_flow     = stock.get_cash_flow(freq='yearly', pretty=False).T

        # Extract the variables we need
        net_income   = financials.get('NetIncome')
        total_assets = balance_sheet.get('TotalAssets')
        equity       = balance_sheet.get('StockholdersEquity')

        # Try two possible column names for dividends
        div_paid = cash_flow.get('CashDividendsPaid')
        if div_paid is None:
            div_paid = cash_flow.get('CommonStockDividendPaid')

        # Skip if any vital variable is missing
        missing = []
        for name, var in [('NetIncome', net_income),
                          ('TotalAssets', total_assets),
                          ('StockholdersEquity', equity),
                          ('DividendsPaid', div_paid)]:
            if var is None:
                missing.append(f"{name} (not found)")
            elif var.dropna().empty:
                missing.append(f"{name} (all NaN)")

        if missing:
            print(f"  Skipping {t} — missing: {', '.join(missing)}")
            continue

        # Build a temp DataFrame for this company
        temp_df = pd.concat([
            net_income.rename('Net_Income'),
            total_assets.rename('Total_Assets'),
            equity.rename('Equity'),
            div_paid.rename('Dividends_Paid')
        ], axis=1, sort=False)

        # Extract year from the date index
        temp_df['Year'] = temp_df.index.year

        # Keep only our target years
        temp_df = temp_df[temp_df['Year'].isin(TARGET_YEARS)]

        if len(temp_df) == 0:
            print(f"  Skipping {t} — no data in target years")
            continue

        # Make dividends positive
        temp_df['Dividends_Paid'] = temp_df['Dividends_Paid'].abs()

        # Replace zeros with NaN
        temp_df['Net_Income']   = temp_df['Net_Income'].replace(0, np.nan)
        temp_df['Total_Assets'] = temp_df['Total_Assets'].replace(0, np.nan)
        temp_df['Equity']       = temp_df['Equity'].replace(0, np.nan)

        # Engineer ratios
        temp_df['DPR']        = temp_df['Dividends_Paid'] / temp_df['Net_Income']
        temp_df['DivYield']   = temp_df['Dividends_Paid'] / temp_df['Total_Assets']
        temp_df['ROE']        = temp_df['Net_Income']     / temp_df['Equity']
        temp_df['Log_Assets'] = np.log(temp_df['Total_Assets'])
        temp_df['Beta']       = np.nan   # filled later
        temp_df['Company']    = t

        print(f"  Success — {len(temp_df)} years")
        all_yf_data.append(temp_df[['Company', 'Year', 'DPR', 'DivYield', 'ROE', 'Log_Assets', 'Beta']])

    except Exception as e:
        print(f"  Error on {t}: {e}")
        continue

# Combine all yfinance companies into one DataFrame
df_yfinance = pd.concat(all_yf_data, ignore_index=True)

print(f"\ndf_yfinance shape: {df_yfinance.shape}")
print(f"Companies: {sorted(df_yfinance['Company'].unique().tolist())}")
print(f"Years: {sorted(df_yfinance['Year'].unique().tolist())}")


Processing AZN...
  Success — 4 years

Processing BAC...
  Success — 4 years

Processing BTI...
  Success — 4 years

Processing HSBC...
  Success — 4 years

Processing JPM...
  Success — 4 years

Processing SHEL...
  Success — 4 years

Processing SO...
  Success — 4 years

Processing UL...
  Success — 4 years

Processing VZ...
  Success — 4 years

Processing WFC...
  Success — 4 years

df_yfinance shape: (40, 7)
Companies: ['AZN', 'BAC', 'BTI', 'HSBC', 'JPM', 'SHEL', 'SO', 'UL', 'VZ', 'WFC']
Years: [2021, 2022, 2023, 2024]


In [22]:
# Cell 8 — Combine SimFin and yfinance into one clean df_final

# Stack the two DataFrames on top of each other
# They have the same columns so this is straightforward
df_combined = pd.concat([
    df_simfin[['Company', 'Year', 'DPR', 'DivYield', 'ROE', 'Log_Assets', 'Beta']],
    df_yfinance[['Company', 'Year', 'DPR', 'DivYield', 'ROE', 'Log_Assets', 'Beta']]
], ignore_index=True)

# Replace infinities with NaN — these come from division by zero
df_combined.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows where any of the core variables are missing
df_combined.dropna(subset=['DPR', 'DivYield', 'ROE', 'Log_Assets'], inplace=True)

# Winsorise extreme outliers at 1st and 99th percentile
# This clips extreme values without removing the observation entirely
for col in ['DPR', 'DivYield', 'ROE']:
    low  = df_combined[col].quantile(0.01)
    high = df_combined[col].quantile(0.99)
    df_combined[col] = df_combined[col].clip(lower=low, upper=high)

# Sort by Company and Year for a clean panel structure
df_combined.sort_values(['Company', 'Year'], inplace=True)
df_combined.reset_index(drop=True, inplace=True)

# This is the final dataset — same format your existing cells expect
df_final = df_combined.copy()

print(f"df_final shape: {df_final.shape}")
print(f"Companies ({df_final['Company'].nunique()}): {sorted(df_final['Company'].unique().tolist())}")
print(f"Years: {sorted(df_final['Year'].unique().tolist())}")
print(f"\nPreview:")
print(df_final.to_string())

df_final shape: (102, 7)
Companies (25): ['AZN', 'BAC', 'BTI', 'CAT', 'CVX', 'D', 'GE', 'HSBC', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'PFE', 'PG', 'SHEL', 'SO', 'T', 'UL', 'VZ', 'WFC', 'WMT', 'XOM']
Years: [2020, 2021, 2022, 2023, 2024]

Preview:
    Company  Year       DPR  DivYield       ROE  Log_Assets  Beta
0       AZN  2022  1.327251  0.045231  0.088776   25.292633   NaN
1       AZN  2023  0.752477  0.044314  0.152134   25.339564   NaN
2       AZN  2024  0.657996  0.044495  0.172486   25.367993   NaN
3       BAC  2022  0.311537  0.002811  0.100762   28.746613   NaN
4       BAC  2023  0.345448  0.002857  0.090195   28.787950   NaN
5       BAC  2024  0.352315  0.002914  0.091756   28.813147   NaN
6       BTI  2022  0.737324  0.032010  0.088446   25.757266   NaN
7       BTI  2023 -0.351848  0.042581 -0.273314   25.500000   NaN
8       BTI  2024  1.699153  0.043844  0.061801   25.501540   NaN
9       CAT  2020  0.748165  0.028637  0.194954   25.084120   NaN
10      CAT  2021

In [23]:
# Cell 9 — Compute time-varying Beta from monthly price data
# Beta measures how much a stock moves relative to the market
# We compute it by regressing each firm's monthly returns against the S&P 500

# ── Step 1: Download S&P 500 benchmark (SPY) ─────────────────────────────────
print("Downloading SPY benchmark...")
spy_raw     = yf.download('SPY', start='2019-01-01', end='2025-01-01', auto_adjust=True)
spy_monthly = spy_raw['Close'].resample('ME').last()
spy_returns = spy_monthly.pct_change().dropna()

# ── Step 2: Download monthly prices for all 25 firms ─────────────────────────
print("Downloading stock prices for all 25 firms...")
all_prices_raw = yf.download(
    df_final['Company'].unique().tolist(),
    start='2019-01-01',
    end='2025-01-01',
    auto_adjust=True
)

# Extract adjusted closing prices and resample to monthly
price_monthly = all_prices_raw['Close'].resample('ME').last()

# Compute monthly returns for all firms at once
stock_returns = price_monthly.pct_change().dropna(how='all')

print(f"Stock returns matrix: {stock_returns.shape}")

# ── Step 3: Compute Beta for each firm-year ───────────────────────────────────
# For each firm and each year, regress the firm's 12 monthly returns
# against SPY's 12 monthly returns — the slope is Beta

beta_dict = {}

for ticker in df_final['Company'].unique():
    if ticker not in stock_returns.columns:
        print(f"  Warning: {ticker} not in price data")
        continue

    for year in df_final['Year'].unique():
        # Get the 12 monthly returns for this firm in this year
        year_mask  = stock_returns.index.year == year
        firm_year  = stock_returns.loc[year_mask, ticker].dropna()
        spy_year   = spy_returns[spy_returns.index.year == year]

        # Align the two series on the same dates
        combined = pd.concat([firm_year, spy_year], axis=1, join='inner')
        combined.columns = ['stock', 'spy']
        combined.dropna(inplace=True)

        # Need at least 6 months of data for a meaningful regression
        if len(combined) < 6:
            continue

        # Run the regression: stock return = alpha + Beta * market return
        slope, intercept, r, p, se = linregress(combined['spy'], combined['stock'])

        # Slope = Beta
        beta_dict[(ticker, year)] = slope

print(f"\nBeta values computed: {len(beta_dict)}")

# ── Step 4: Map Beta onto df_final ───────────────────────────────────────────
df_final['Beta'] = df_final.apply(
    lambda row: beta_dict.get((row['Company'], row['Year']), np.nan),
    axis=1
)

print(f"Beta assigned: {df_final['Beta'].notna().sum()} of {len(df_final)} rows")
print(f"Missing Beta : {df_final['Beta'].isna().sum()} rows")
print(f"\nSample Beta values:")
print(df_final[['Company', 'Year', 'Beta']].head(15).to_string())

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  24 of 25 completed


Stock returns matrix: (71, 25)

Beta values computed: 125
Beta assigned: 102 of 102 rows
Missing Beta : 0 rows

Sample Beta values:
   Company  Year      Beta
0      AZN  2022  0.679562
1      AZN  2023  0.259717
2      AZN  2024 -0.408741
3      BAC  2022  1.256388
4      BAC  2023  1.426719
5      BAC  2024  1.239263
6      BTI  2022  0.204021
7      BTI  2023  0.533686
8      BTI  2024  0.945486
9      CAT  2020  0.502507
10     CAT  2021  1.226848
11     CAT  2022  1.798736
12     CAT  2023  1.702860
13     CAT  2024  1.806129
14     CVX  2020  1.757851


In [25]:
# Cell 10 — VIF Diagnostic (Multicollinearity Check)
# VIF tells us whether any explanatory variables are too correlated with each other
# A VIF below 5 is acceptable, below 2 is excellent

# Add a constant column (intercept) — required for VIF calculation
X_vif = sm.add_constant(df_final[['DPR', 'DivYield', 'ROE', 'Beta']])

# Compute VIF for each variable
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF']      = [variance_inflation_factor(X_vif.values, i)
                        for i in range(X_vif.shape[1])]

# Label each variable's result
def vif_label(v):
    if v >= 10:
        return 'SEVERE'
    elif v >= 5:
        return 'MODERATE'
    else:
        return 'OK'

vif_data['Assessment'] = vif_data['VIF'].apply(vif_label)

print("=" * 40)
print("       VIF RESULTS")
print("=" * 40)
print(vif_data[vif_data['Variable'] != 'const'].to_string(index=False))
print("=" * 40)
print("Threshold: VIF < 5 = OK, 5-10 = MODERATE, >10 = SEVERE")

       VIF RESULTS
Variable      VIF Assessment
     DPR 1.072231         OK
DivYield 1.081549         OK
     ROE 1.041278         OK
    Beta 1.049617         OK
Threshold: VIF < 5 = OK, 5-10 = MODERATE, >10 = SEVERE


In [26]:
# Cell 11 — Pooled OLS (baseline regression)
# This treats all 102 observations as if they were independent
# It is our baseline — we expect panel estimators to improve on it

# Step 1: Define dependent variable (Y) and explanatory variables (X)
Y = df_final['Log_Assets']
X = sm.add_constant(df_final[['DPR', 'DivYield', 'ROE', 'Beta']])

# Step 2: Run OLS with firm-level clustered standard errors
# Clustering corrects for the fact that observations from the same firm
# are not truly independent of each other
pooled_ols = sm.OLS(Y, X).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_final['Company']}
)

print(pooled_ols.summary())

                            OLS Regression Results                            
Dep. Variable:             Log_Assets   R-squared:                       0.514
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     10.65
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           4.16e-05
Time:                        17:59:49   Log-Likelihood:                -121.02
No. Observations:                 102   AIC:                             252.0
Df Residuals:                      97   BIC:                             265.2
Df Model:                           4                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         27.5003      0.405     67.880      0.0

In [29]:
# Cell 12 — Panel MultiIndex and Fixed Effects
df_panel = df_final.set_index(['Company', 'Year'])

print("Panel structure:")
print(f"  Index levels : {df_panel.index.names}")
print(f"  Shape        : {df_panel.shape}")

fe_model = PanelOLS(
    dependent=df_panel['Log_Assets'],
    exog=df_panel[['DPR', 'DivYield', 'ROE', 'Beta']],
    entity_effects=True,
    drop_absorbed=True
)

fe_results = fe_model.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(fe_results.summary)

Panel structure:
  Index levels : ['Company', 'Year']
  Shape        : (102, 5)
                          PanelOLS Estimation Summary                           
Dep. Variable:             Log_Assets   R-squared:                        0.0756
Estimator:                   PanelOLS   R-squared (Between):             -0.0162
No. Observations:                 102   R-squared (Within):               0.0756
Date:                Thu, Jun 04 2026   R-squared (Overall):             -0.0176
Time:                        18:03:13   Log-likelihood                    110.42
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.4930
Entities:                          25   P-value                           0.2132
Avg Obs:                       4.0800   Distribution:                    F(4,73)
Min Obs:                       3.0000                                           
Max Obs:                     

In [30]:
# Cell 13 — Random Effects Estimator
# Random Effects assumes firm-level unobserved effects are uncorrelated
# with the explanatory variables — more efficient than FE if that holds

re_model = RandomEffects(
    dependent=df_panel['Log_Assets'],
    exog=df_panel[['DPR', 'DivYield', 'ROE', 'Beta']]
)

re_results = re_model.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(re_results.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:             Log_Assets   R-squared:                        0.0102
Estimator:              RandomEffects   R-squared (Between):             -0.0083
No. Observations:                 102   R-squared (Within):               0.0641
Date:                Thu, Jun 04 2026   R-squared (Overall):             -0.0090
Time:                        18:04:41   Log-likelihood                    37.276
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.2518
Entities:                          25   P-value                           0.9080
Avg Obs:                       4.0800   Distribution:                    F(4,98)
Min Obs:                       3.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             1.2167
                            

In [32]:
# Cell 14 — Hausman Test
fe_params = fe_results.params[['DPR', 'DivYield', 'ROE', 'Beta']]
re_params = re_results.params[['DPR', 'DivYield', 'ROE', 'Beta']]

diff = fe_params - re_params

fe_cov = fe_results.cov.loc[['DPR', 'DivYield', 'ROE', 'Beta'],
                             ['DPR', 'DivYield', 'ROE', 'Beta']]
re_cov = re_results.cov.loc[['DPR', 'DivYield', 'ROE', 'Beta'],
                             ['DPR', 'DivYield', 'ROE', 'Beta']]

cov_diff     = fe_cov - re_cov
hausman_stat = np.dot(np.dot(diff.values, np.linalg.inv(cov_diff.values)), diff.values)
hausman_df   = len(diff)
hausman_p    = stats.chi2.sf(hausman_stat, df=hausman_df)

print("=" * 45)
print("         HAUSMAN TEST RESULTS")
print("=" * 45)
print(f"  Chi-squared statistic : {hausman_stat:.4f}")
print(f"  Degrees of freedom    : {hausman_df}")
print(f"  P-value               : {hausman_p:.4f}")
print("=" * 45)
if hausman_p < 0.05:
    print("  CONCLUSION: Reject RE → Use Fixed Effects")
else:
    print("  CONCLUSION: Fail to reject RE → Use Random Effects")
print("=" * 45)

         HAUSMAN TEST RESULTS
  Chi-squared statistic : 2.3309
  Degrees of freedom    : 4
  P-value               : 0.6752
  CONCLUSION: Fail to reject RE → Use Random Effects


In [33]:
# Cell 15 — Save dataset and robustness check with Log Market Cap

# Save the final dataset to CSV
df_final.to_csv('dividend_panel_data.csv', index=False)
print(f"Saved dividend_panel_data.csv — {df_final.shape[0]} rows")

# ── Robustness Check: Log Market Cap as dependent variable ───────────────────
# Fetch current market cap for each firm from yfinance
print("\nFetching market cap data...")

mcap_dict = {}
for t in df_final['Company'].unique():
    stock = yf.Ticker(t)
    mcap  = stock.info.get('marketCap', None)
    if mcap and mcap > 0:
        mcap_dict[t] = np.log(mcap)
    else:
        mcap_dict[t] = np.nan

# Map log market cap onto df_final
df_final['Log_MarketCap'] = df_final['Company'].map(mcap_dict)

print(f"Missing Log_MarketCap: {df_final['Log_MarketCap'].isna().sum()} rows")

# Run cross-sectional OLS using Log Market Cap as dependent variable
Y_mc = df_final['Log_MarketCap'].dropna()
X_mc = sm.add_constant(
    df_final.loc[Y_mc.index, ['DPR', 'DivYield', 'ROE', 'Beta']]
)

robustness_ols = sm.OLS(Y_mc, X_mc).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_final.loc[Y_mc.index, 'Company']}
)

print("\nRobustness Check — Log Market Cap as Dependent Variable:")
print(robustness_ols.summary())

Saved dividend_panel_data.csv — 102 rows

Fetching market cap data...
Missing Log_MarketCap: 0 rows

Robustness Check — Log Market Cap as Dependent Variable:
                            OLS Regression Results                            
Dep. Variable:          Log_MarketCap   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.026
Method:                 Least Squares   F-statistic:                     3.317
Date:                Thu, 04 Jun 2026   Prob (F-statistic):             0.0268
Time:                        18:07:12   Log-Likelihood:                -100.54
No. Observations:                 102   AIC:                             211.1
Df Residuals:                      97   BIC:                             224.2
Df Model:                           4                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|